In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
test_ids = test['Index']

In [4]:
train.drop('Index', axis=1, inplace=True)
test.drop('Index', axis=1, inplace=True)

In [5]:
for df in [train, test]:

    df[['hour', 'minute']] = (
        df['timestamp']
        .str.split(':', expand=True)
        .astype(int)
    )

    df['is_peak_hour'] = (
        df['hour'].isin([7,8,9,17,18,19])
    ).astype(int)

    df['is_night'] = (
        ((df['hour'] >= 22) |
         (df['hour'] <= 5))
    ).astype(int)

    df['hour_bin'] = pd.cut(
        df['hour'],
        bins=[0,6,12,18,24],
        labels=False,
        include_lowest=True
    )

In [7]:
train['RoadType'] = train['RoadType'].fillna(
    train['RoadType'].mode()[0]
)

In [8]:
train['Weather'].fillna(
    train['Weather'].mode()[0],
    inplace=True
)

test['Weather'].fillna(
    train['Weather'].mode()[0],
    inplace=True
)

C:\Users\mysti\AppData\Local\Temp\ipykernel_9988\750883540.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Weather'].fillna(
C:\Users\mysti\AppData\Local\Temp\ipykernel_9988\750883540.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].me

In [9]:
train['Temperature'].fillna(
    train['Temperature'].median(),
    inplace=True
)

test['Temperature'].fillna(
    train['Temperature'].median(),
    inplace=True
)

C:\Users\mysti\AppData\Local\Temp\ipykernel_9988\2305969208.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Temperature'].fillna(
C:\Users\mysti\AppData\Local\Temp\ipykernel_9988\2305969208.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[c

In [10]:
cat_cols = [
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

In [11]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le.fit(combined)

    train[col] = le.transform(train[col])

    test[col] = le.transform(test[col])

In [12]:
geo_mean = train.groupby(
    'geohash'
)['demand'].mean()

In [13]:
train['geohash_te'] = (
    train['geohash']
    .map(geo_mean)
)

test['geohash_te'] = (
    test['geohash']
    .map(geo_mean)
)

In [14]:
global_mean = train['demand'].mean()

test['geohash_te'].fillna(
    global_mean,
    inplace=True
)

C:\Users\mysti\AppData\Local\Temp\ipykernel_9988\3316517688.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test['geohash_te'].fillna(


In [15]:
for df in [train, test]:

    df['hour_sin'] = np.sin(
        2 * np.pi * df['hour'] / 24
    )

    df['hour_cos'] = np.cos(
        2 * np.pi * df['hour'] / 24
    )

    df['minute_sin'] = np.sin(
        2 * np.pi * df['minute'] / 60
    )

    df['minute_cos'] = np.cos(
        2 * np.pi * df['minute'] / 60
    )

In [16]:
for df in [train, test]:

    df['lane_pressure'] = (
        df['NumberofLanes']
        /
        (df['RoadType'] + 1)
    )

    df['vehicle_lane_interaction'] = (
        df['LargeVehicles']
        *
        df['NumberofLanes']
    )

    df['temp_hour_interaction'] = (
        df['Temperature']
        *
        df['hour']
    )

    df['day_hour_interaction'] = (
        df['day']
        *
        df['hour']
    )

    df['weather_hour_interaction'] = (
        df['Weather']
        *
        df['hour']
    )

In [17]:
train.drop(
    ['geohash', 'timestamp'],
    axis=1,
    inplace=True
)

test.drop(
    ['geohash', 'timestamp'],
    axis=1,
    inplace=True
)

In [18]:
X = train.drop('demand', axis=1)

y = train['demand']

In [19]:
rf_model = RandomForestRegressor(

    n_estimators=300,

    max_depth=20,

    min_samples_leaf=2,

    random_state=42,

    n_jobs=-1
)

rf_model.fit(X, y)

,n_estimators,300
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
rf_preds = rf_model.predict(test)

In [21]:
et_model = ExtraTreesRegressor(

    n_estimators=400,

    max_depth=25,

    min_samples_leaf=2,

    random_state=42,

    n_jobs=-1
)

et_model.fit(X, y)

,n_estimators,400
,criterion,'squared_error'
,max_depth,25
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [22]:
et_preds = et_model.predict(test)

In [23]:
final_preds = (
    0.75 * rf_preds
    +
    0.25 * et_preds
)

In [24]:
submission = pd.DataFrame({

    'Index': test_ids,

    'demand': final_preds
})

In [25]:
submission.to_csv(
    'submission_final.csv',
    index=False
)

In [34]:
print(train.columns.tolist())

['day', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'is_peak_hour', 'is_night', 'hour_bin', 'geohash_te', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lane_pressure', 'vehicle_lane_interaction', 'temp_hour_interaction', 'day_hour_interaction', 'weather_hour_interaction']


In [35]:
original_train = pd.read_csv('train.csv')
original_test = pd.read_csv('test.csv')

In [36]:
print("Train:", original_train.shape)
print("Test:", original_test.shape)

print("Unique geohash:", original_train['geohash'].nunique())
print("Unique day:", original_train['day'].nunique())
print("Unique timestamp:", original_train['timestamp'].nunique())

Train: (77299, 11)
Test: (41778, 10)
Unique geohash: 1249
Unique day: 2
Unique timestamp: 96


In [37]:
print(
    original_train[
        ['geohash','day','timestamp']
    ].duplicated().sum()
)

0


In [38]:
print(
    original_train.groupby(
        ['geohash','day','timestamp']
    )['demand'].nunique().max()
)

1


In [39]:
print(sorted(original_train['day'].unique()))
print(sorted(original_test['day'].unique()))

[np.int64(48), np.int64(49)]
[np.int64(49)]


In [40]:
set(original_test['day']) - set(original_train['day'])

set()

In [41]:
train_keys = set(
    zip(
        original_train['geohash'],
        original_train['day'],
        original_train['timestamp']
    )
)

test_keys = set(
    zip(
        original_test['geohash'],
        original_test['day'],
        original_test['timestamp']
    )
)

overlap = len(train_keys & test_keys)

print("Overlap:", overlap)
print("Test rows:", len(test_keys))
print("Overlap %:", overlap / len(test_keys) * 100)

Overlap: 0
Test rows: 41778
Overlap %: 0.0


In [42]:
train_geo = set(original_train['geohash'])
test_geo = set(original_test['geohash'])

print("Common geohashes:", len(train_geo & test_geo))
print("Test geohashes:", len(test_geo))

Common geohashes: 1180
Test geohashes: 1190


In [43]:
pivot = original_train.pivot_table(
    index='geohash',
    columns=['day','timestamp'],
    values='demand'
)

print(pivot.shape)

(1249, 105)


In [44]:
geo_stats = original_train.groupby('geohash')['demand'].agg(
    ['mean','std']
)

print(geo_stats['std'].describe())

count    1217.000000
mean        0.038588
std         0.050859
min         0.000087
25%         0.010641
50%         0.022047
75%         0.043653
max         0.376928
Name: std, dtype: float64


In [45]:
print(
    original_train.groupby('geohash')['demand']
    .mean()
    .sort_values()
    .head()
)

print(
    original_train.groupby('geohash')['demand']
    .mean()
    .sort_values()
    .tail()
)

geohash
qp03zy    0.000495
qp08bt    0.000780
qp09k7    0.000793
qp093h    0.000815
qp09bv    0.000922
Name: demand, dtype: float64
geohash
qp096x    0.665630
qp09d8    0.669318
qp09e5    0.864989
qp09ft    0.868850
qp09d9    0.960715
Name: demand, dtype: float64


In [46]:
check = (
    original_train
    .groupby(['geohash','timestamp'])
    .size()
)

print(check.describe())


count    70876.000000
mean         1.090623
std          0.287074
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
dtype: float64


In [47]:
print(check.value_counts().head(10))

1    64453
2     6423
Name: count, dtype: int64


In [48]:
original_train.groupby('RoadType')['demand'].mean().sort_values()

RoadType
Residential    0.057209
Street         0.273164
Highway        0.610756
Name: demand, dtype: float64

In [49]:
original_train.groupby('Weather')['demand'].mean().sort_values()

Weather
Snowy    0.092581
Foggy    0.093372
Sunny    0.094247
Rainy    0.094471
Name: demand, dtype: float64

In [50]:
original_train.groupby('NumberofLanes')['demand'].mean().sort_values()

NumberofLanes
2    0.077488
3    0.077859
1    0.088104
4    0.602882
5    0.607556
Name: demand, dtype: float64

In [51]:
pd.crosstab(
    original_train['RoadType'],
    original_train['NumberofLanes']
)


NumberofLanes,1,2,3,4,5
RoadType,,,,,
Highway,0,855,881,915,909
Residential,23269,23091,22870,0,0
Street,3909,0,0,0,0


In [52]:
original_train.groupby(
    ['RoadType','NumberofLanes']
)['demand'].mean()

RoadType     NumberofLanes
Highway      2                0.619695
             3                0.613320
             4                0.603511
             5                0.607156
Residential  1                0.057050
             2                0.057326
             3                0.057253
Street       1                0.273164
Name: demand, dtype: float64

In [53]:
geo_road = pd.crosstab(
    original_train['geohash'],
    original_train['RoadType']
)

print(
    (geo_road > 0).sum(axis=1).value_counts()
)

1    994
2    128
3    127
Name: count, dtype: int64


In [54]:
original_train.groupby('timestamp')['demand'].mean()

timestamp
0:0     0.081056
0:15    0.081929
0:30    0.084357
0:45    0.085994
10:0    0.110319
          ...   
8:45    0.105793
9:0     0.107004
9:15    0.112049
9:30    0.109178
9:45    0.108777
Name: demand, Length: 96, dtype: float64

In [55]:
original_train.groupby('timestamp')['demand'].mean().sort_values().head(10)

original_train.groupby('timestamp')['demand'].mean().sort_values().tail(10)

timestamp
12:15    0.114676
11:0     0.115395
13:15    0.116641
13:45    0.117046
11:45    0.117129
11:30    0.117561
12:0     0.117790
14:0     0.117821
11:15    0.119174
13:30    0.119193
Name: demand, dtype: float64

In [56]:
geo_road = pd.crosstab(...)

TypeError: crosstab() missing 1 required positional argument: 'columns'

In [57]:
groupby('timestamp')['demand'].mean()

NameError: name 'groupby' is not defined

In [58]:
print(original_train['demand'].describe())

count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [59]:
print(original_train['demand'].nunique())

76715


In [60]:
original_train['demand'].describe()
original_train['demand'].nunique()

76715

In [61]:
original_train = pd.read_csv('train.csv')
original_test = pd.read_csv('test.csv')

In [62]:
geo_mean = (
    original_train
    .groupby('geohash')['demand']
    .mean()
)

In [63]:
road_mean = (
    original_train
    .groupby('RoadType')['demand']
    .mean()
)

In [64]:
road_lane_mean = (
    original_train
    .groupby(
        ['RoadType','NumberofLanes']
    )['demand']
    .mean()
)

In [65]:
time_mean = (
    original_train
    .groupby('timestamp')['demand']
    .mean()
)

In [66]:
original_train['geo_mean'] = (
    original_train['geohash']
    .map(geo_mean)
)

original_test['geo_mean'] = (
    original_test['geohash']
    .map(geo_mean)
)

In [67]:
original_train['road_mean'] = (
    original_train['RoadType']
    .map(road_mean)
)

original_test['road_mean'] = (
    original_test['RoadType']
    .map(road_mean)
)

In [68]:
original_train['road_lane_mean'] = (
    original_train
    .set_index(
        ['RoadType','NumberofLanes']
    )
    .index
    .map(road_lane_mean)
)

original_test['road_lane_mean'] = (
    original_test
    .set_index(
        ['RoadType','NumberofLanes']
    )
    .index
    .map(road_lane_mean)
)

In [69]:
original_train['time_mean'] = (
    original_train['timestamp']
    .map(time_mean)
)

original_test['time_mean'] = (
    original_test['timestamp']
    .map(time_mean)
)

In [70]:
print(
    original_train[
        [
            'geo_mean',
            'road_mean',
            'road_lane_mean',
            'time_mean'
        ]
    ].isnull().sum()
)

print(
    original_test[
        [
            'geo_mean',
            'road_mean',
            'road_lane_mean',
            'time_mean'
        ]
    ].isnull().sum()
)

geo_mean            0
road_mean         600
road_lane_mean    600
time_mean           0
dtype: int64
geo_mean           25
road_mean         324
road_lane_mean    324
time_mean           0
dtype: int64


In [71]:
overall_mean = original_train['demand'].mean()

for col in [
    'geo_mean',
    'road_mean',
    'road_lane_mean',
    'time_mean'
]:
    original_train[col] = original_train[col].fillna(overall_mean)
    original_test[col] = original_test[col].fillna(overall_mean)

In [72]:
print(
    original_train[
        ['geo_mean','road_mean','road_lane_mean','time_mean']
    ].isnull().sum()
)

print(
    original_test[
        ['geo_mean','road_mean','road_lane_mean','time_mean']
    ].isnull().sum()
)

geo_mean          0
road_mean         0
road_lane_mean    0
time_mean         0
dtype: int64
geo_mean          0
road_mean         0
road_lane_mean    0
time_mean         0
dtype: int64


In [73]:
print(original_train.columns.tolist())

['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'geo_mean', 'road_mean', 'road_lane_mean', 'time_mean']


In [74]:
print(train.columns.tolist())

['day', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'is_peak_hour', 'is_night', 'hour_bin', 'geohash_te', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lane_pressure', 'vehicle_lane_interaction', 'temp_hour_interaction', 'day_hour_interaction', 'weather_hour_interaction']


In [75]:
train['geo_mean'] = original_train['geohash'].map(
    original_train.groupby('geohash')['demand'].mean()
)

test['geo_mean'] = original_test['geohash'].map(
    original_train.groupby('geohash')['demand'].mean()
)

In [76]:
train['road_mean'] = original_train['RoadType'].map(
    original_train.groupby('RoadType')['demand'].mean()
)

test['road_mean'] = original_test['RoadType'].map(
    original_train.groupby('RoadType')['demand'].mean()
)

In [77]:
road_lane_mean = (
    original_train
    .groupby(['RoadType','NumberofLanes'])['demand']
    .mean()
)

train['road_lane_mean'] = list(
    zip(
        original_train['RoadType'],
        original_train['NumberofLanes']
    )
)

train['road_lane_mean'] = train['road_lane_mean'].map(
    road_lane_mean
)

test['road_lane_mean'] = list(
    zip(
        original_test['RoadType'],
        original_test['NumberofLanes']
    )
)

test['road_lane_mean'] = test['road_lane_mean'].map(
    road_lane_mean
)

In [78]:
time_mean = (
    original_train
    .groupby('timestamp')['demand']
    .mean()
)

train['time_mean'] = original_train['timestamp'].map(
    time_mean
)

test['time_mean'] = original_test['timestamp'].map(
    time_mean
)

In [79]:
overall_mean = original_train['demand'].mean()

for col in [
    'geo_mean',
    'road_mean',
    'road_lane_mean',
    'time_mean'
]:
    train[col] = train[col].fillna(overall_mean)
    test[col] = test[col].fillna(overall_mean)

In [80]:
print(train.columns.tolist())

['day', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'is_peak_hour', 'is_night', 'hour_bin', 'geohash_te', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lane_pressure', 'vehicle_lane_interaction', 'temp_hour_interaction', 'day_hour_interaction', 'weather_hour_interaction', 'geo_mean', 'road_mean', 'road_lane_mean', 'time_mean']


In [81]:
print(train.shape)
print(test.shape)

(77299, 27)
(41778, 26)


In [82]:
features = [col for col in train.columns if col != 'demand']

X = train[features]
y = train['demand']

X_test = test[features]

In [85]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

rf_preds = rf.predict(X_test)

In [84]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=25,
    random_state=42,
    n_jobs=-1
)

et.fit(X, y)

et_preds = et.predict(X_test)

In [86]:
final_preds = (
    0.75 * rf_preds +
    0.25 * et_preds
)

In [87]:
submission = pd.DataFrame({
    'Index': test.index,
    'demand': final_preds
})

submission.to_csv(
    'submission_geo_stats.csv',
    index=False
)

In [88]:
submission.head()

,Index,demand
0,0,0.043162
1,1,0.029954
2,2,0.039778
3,3,0.044679
4,4,0.058290


In [89]:
submission.shape

(41778, 2)

In [90]:
submission = pd.DataFrame({
    'Index': range(len(test)),
    'demand': final_preds
})

submission.to_csv('submission_geo_stats.csv', index=False)

print(submission.head())
print(submission.shape)

   Index    demand
0      0  0.043162
1      1  0.029954
2      2  0.039778
3      3  0.044679
4      4  0.058290
(41778, 2)
